In [54]:
import html

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"

In [55]:
from datasets import load_dataset

In [56]:
data_files = {
    "train": f"zip://drugsComTrain_raw.tsv::simplecache::{url}",
    "test": f"zip://drugsComTest_raw.tsv::simplecache::{url}",
}

In [57]:
dataset = load_dataset(
    'csv',
    data_files=data_files,
    delimiter="\t"
)

In [58]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [59]:
drug_sample = dataset['train'].shuffle(seed=42).select(range(1000))
drug_sample[:5]

{'Unnamed: 0': [87571, 178045, 80482, 159268, 205477],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic', 'TriNessa', 'Pristiq'],
 'condition': ['Gout, Acute',
  'ibromyalgia',
  'Inflammatory Conditions',
  'Birth Control',
  'Depression'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',

In [60]:
for split in dataset.keys():
    assert len(dataset[split]) == len(dataset[split].unique('Unnamed: 0'))

In [61]:
dataset = dataset.rename_column('Unnamed: 0', 'patient_id')
dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [62]:
len(dataset['train'].unique('drugName'))

3436

In [63]:
def lowercase_condition(example):
    return {'condition': example['condition'].lower()}

In [64]:
dataset.map(lowercase_condition)

Map:   0%|          | 30/161297 [00:00<00:44, 3636.26 examples/s]


AttributeError: 'NoneType' object has no attribute 'lower'

In [65]:
def filter_nones(x):
    return x['condition'] is not None

In [66]:
dataset = dataset.filter(filter_nones)

Filter: 100%|██████████| 53766/53766 [00:00<00:00, 461736.97 examples/s]


In [67]:
dataset = dataset.map(lowercase_condition)

Map: 100%|██████████| 53471/53471 [00:02<00:00, 19049.15 examples/s]


In [68]:
len(dataset['train'].unique('condition'))

884

In [34]:
"""
Whenever you’re dealing with customer reviews, a good practice is to check the number of words in each review. A review might be just a single word like “Great!” or a full-blown essay with thousands of words, and depending on the use case you’ll need to handle these extremes differently. To compute the number of words in each review, we’ll use a rough heuristic based on splitting each text by whitespace.
"""

'\nWhenever you’re dealing with customer reviews, a good practice is to check the number of words in each review. A review might be just a single word like “Great!” or a full-blown essay with thousands of words, and depending on the use case you’ll need to handle these extremes differently. To compute the number of words in each review, we’ll use a rough heuristic based on splitting each text by whitespace.\n'

In [69]:
def compute_review_len(example):
    return {'review_length': len(example['review'].split())}

In [70]:
dataset = dataset.map(compute_review_len)
dataset['train'][0]

Map: 100%|██████████| 53471/53471 [00:02<00:00, 23945.61 examples/s]


{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [71]:
dataset['train'].sort('review_length', reverse=False)[:5]


C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\pyarrow\compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


{'patient_id': [111469, 13653, 53602, 135265, 160223],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse',
  'Zegerid',
  'Rivaroxaban'],
 'condition': ['hepatitis c',
  'adhd',
  'birth control',
  'gerd',
  'deep vein thrombosis'],
 'review': ['"Headache"', '"Great"', '"Awesome"', '"expensive"', '"Good"'],
 'rating': [10.0, 10.0, 10.0, 10.0, 9.0],
 'date': ['February 3, 2015',
  'October 20, 2009',
  'November 23, 2015',
  'March 12, 2008',
  'December 8, 2013'],
 'usefulCount': [41, 3, 0, 32, 11],
 'review_length': [1, 1, 1, 1, 1]}

In [72]:
dataset = dataset.filter(lambda x: x['review_length'] > 30)
dataset.num_rows, dataset.num_columns

Filter: 100%|██████████| 53471/53471 [00:00<00:00, 507506.29 examples/s]


({'train': 138514, 'test': 46108}, {'train': 8, 'test': 8})

In [73]:
sorted_reviews = dataset["train"].sort("review_length", reverse=True)

# Inspect the 3 longest reviews
for example in sorted_reviews.select(range(3)):
    print(example["review_length"])
    print(example["review"])
    print()

1894
"Two and a half months ago I was prescribed Venlafaxine to help prevent chronic migraines.
It did help the migraines (reduced them by almost half), but with it came a host of side effects that were far worse than the problem I was trying to get rid of.
Having now come off of the stuff, I would not recommend anyone ever use Venlafaxine unless they suffer from extreme / suicidal depression. I mean extreme in the most emphatic sense of the word. 
Before trying Venlafaxine, I was a writer. While on Venlafaxine, I could barely write or speak or communicate at all. More than that, I just didn&#039;t want to. Not normal for a usually outgoing extrovert.
Now, I&#039;m beginning to write again - but my ability to speak and converse with others has deteriorated by about 95%. Writing these words is taking forever; keeping up in conversation with even one person is impossible, and I barely see the point of trying either. On Venlafaxine, words pretty much left me - my conversational vocabulary

In [74]:
import html

In [75]:
dataset = dataset.map(lambda x: {'review': html.unescape(x['review'])})

Map: 100%|██████████| 46108/46108 [00:02<00:00, 17614.53 examples/s]


In [76]:
new_dataset = dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]}, batched=True
)

Map: 100%|██████████| 46108/46108 [00:00<00:00, 699222.54 examples/s]


In [77]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

In [78]:
def tokenize(example):
    return tokenizer(example['review'], truncation=True, max_length=128, return_overflowing_tokens=True)

In [82]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map: 100%|██████████| 46108/46108 [00:14<00:00, 3174.36 examples/s]


### Dataframes + Datasets

In [90]:
dataset.set_format('pandas')

In [92]:
dataset['train'][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


In [93]:
train_df = dataset['train'][:]

In [94]:
frequencies = (
    train_df['condition'].value_counts().to_frame().reset_index().rename(columns={'index': 'condition', 'count': 'frequency'})
)

In [95]:
frequencies.head()

,condition,frequency
0,birth control,27655
1,depression,8023
2,acne,5209
3,anxiety,4991
4,pain,4744


In [96]:
from datasets import Dataset

In [99]:
freq_dataset = Dataset.from_pandas(frequencies)
freq_dataset

Dataset({
    features: ['condition', 'frequency'],
    num_rows: 819
})

In [100]:
drug_dataset_clean = dataset['train'].train_test_split(train_size=0.8, seed=42)
drug_dataset_clean['validation'] = drug_dataset_clean.pop('test')
drug_dataset_clean['test'] = dataset['test']
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

### Saving dataset

In [101]:
drug_dataset_clean.save_to_disk('drug-reviews')

Saving the dataset (1/1 shards): 100%|██████████| 46108/46108 [00:00<00:00, 657367.58 examples/s]


In [102]:
from datasets import load_from_disk

drug_dataset_reloaded = load_from_disk('drug-reviews')
drug_dataset_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [103]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format: 100%|██████████| 47/47 [00:00<00:00, 234.21ba/s]
